In [1]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\| **Quickstart** \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| [Build
Model](buildmodel_tutorial.html) \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Quickstart
==========

This section runs through the API for common tasks in machine learning.
Refer to the links in each section to dive deeper.

Working with data
-----------------

PyTorch has two [primitives to work with
data](https://pytorch.org/docs/stable/data.html):
`torch.utils.data.DataLoader` and `torch.utils.data.Dataset`. `Dataset`
stores the samples and their corresponding labels, and `DataLoader`
wraps an iterable around the `Dataset`.


In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

PyTorch offers domain-specific libraries such as
[TorchText](https://pytorch.org/text/stable/index.html),
[TorchVision](https://pytorch.org/vision/stable/index.html), and
[TorchAudio](https://pytorch.org/audio/stable/index.html), all of which
include datasets. For this tutorial, we will be using a TorchVision
dataset.

The `torchvision.datasets` module contains `Dataset` objects for many
real-world vision data like CIFAR, COCO ([full list
here](https://pytorch.org/vision/stable/datasets.html)). In this
tutorial, we use the FashionMNIST dataset. Every TorchVision `Dataset`
includes two arguments: `transform` and `target_transform` to modify the
samples and labels respectively.


In [3]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data", # Carpeta local donde se almacenarán los datos descargados
    train=True, # Indica que se desea descargar el conjunto de datos de entrenamiento
    download=True, # Descarga los datos si no están presentes en la carpeta especificada
    transform=ToTensor(), # Aplica una transformación a los datos, en este caso convierte las imágenes a tensores   
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False, # Indica que se desea descargar el conjunto de datos de prueba (test)
    download=True,
    transform=ToTensor(),
)

We pass the `Dataset` as an argument to `DataLoader`. This wraps an
iterable over our dataset, and supports automatic batching, sampling,
shuffling and multiprocess data loading. Here we define a batch size of
64, i.e. each element in the dataloader iterable will return a batch of
64 features and labels.


In [4]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

print(f'Type of train_dataloader: {type(train_dataloader)}')
print(f'Type of test_dataloader: {type(test_dataloader)}')

# Iterate through the test dataloader to inspect the shape of the data
"""
X contiene las imágenes del dataset FashionMNIST:

Son tensores de forma [64, 1, 28, 28] donde:
64 = tamaño del lote (batch_size)
1 = 1 canal (imágenes en escala de grises)
28x28 = dimensiones de cada imagen en píxeles

y contiene las etiquetas (clases) correspondientes a cada imagen:

Son números del 0 al 9 representando las 10 categorías de ropa.

"""
for X, y in test_dataloader:
    # Print the shape of the input tensor X [N=batch_size, C=channels, H=height, W=width]
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    # Print the shape and data type of the labels tensor y
    print(f"Shape of y: {y.shape}, type of y: {y.dtype}")
    # Exit the loop after the first batch to avoid unnecessary iteration
    break

Type of train_dataloader: <class 'torch.utils.data.dataloader.DataLoader'>
Type of test_dataloader: <class 'torch.utils.data.dataloader.DataLoader'>
Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]), type of y: torch.int64


Read more about [loading data in PyTorch](data_tutorial.html).


------------------------------------------------------------------------


Creating Models
===============

To define a neural network in PyTorch, we create a class that inherits
from
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
We define the layers of the network in the `__init__` function and
specify how data will pass through the network in the `forward`
function. To accelerate operations in the neural network, we move it to
the
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
# Definir la clase NeuralNetwork que hereda de nn.Module. 
# OJO:No se ejecuta nada, solo se define la clase (y sus métodos) para luego crear una instancia del modelo
class NeuralNetwork(nn.Module):
    # Constructor de la clase - inicializa las capas de la red neuronal
    def __init__(self):
        # Llama al constructor de la clase padre (nn.Module)
        super().__init__()
        # Capa que aplana la imagen 2D en un vector 1D (28x28 -> 784)
        self.flatten = nn.Flatten()
        # Stack secuencial de capas lineales y de activación
        self.linear_relu_stack = nn.Sequential(
            # Primera capa lineal: 784 entradas (28*28 píxeles) -> 512 neuronas
            nn.Linear(28*28, 512),
            # Función de activación ReLU para introducir no linealidad
            nn.ReLU(),
            # Segunda capa lineal: 512 entradas -> 512 neuronas (capa oculta)
            nn.Linear(512, 512),
            # Segunda función de activación ReLU
            nn.ReLU(),
            # Capa de salida: 512 entradas -> 10 salidas logit (una por cada clase de ropa)
            # Un logit es la salida “cruda” de la última capa lineal del modelo, puede ser cualquier número real.
            # Para convertir logits en probabilidades se aplica Softmax
            nn.Linear(512, 10)
        )

    # Método forward: define cómo fluyen los datos a través de la red
    def forward(self, x):
        # Aplana el tensor de entrada (de [batch_size, 1, 28, 28] a [batch_size, 784])
        x = self.flatten(x)
        # Pasa los datos aplanados por el stack de capas (lineales + ReLU)
        logits = self.linear_relu_stack(x)
        # Retorna los logits (valores sin normalizar antes de aplicar softmax)
        return logits

# Crea una instancia del modelo y la mueve al dispositivo (CPU o GPU)
model = NeuralNetwork().to(device)

# Imprime la arquitectura completa del modelo
print(model)    

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Read more about [building neural networks in
PyTorch](buildmodel_tutorial.html).


------------------------------------------------------------------------


Optimizing the Model Parameters
===============================

To train a model, we need a [loss
function](https://pytorch.org/docs/stable/nn.html#loss-functions) and an
[optimizer](https://pytorch.org/docs/stable/optim.html).


In [6]:
# Define la función de pérdida: CrossEntropy para clasificación multiclase
loss_fn = nn.CrossEntropyLoss()

# Iterate through model parameters and display them
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
print(model.parameters().__next__())

# Define el optimizador: SGD (Stochastic Gradient Descent) con learning rate de 0.001
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

linear_relu_stack.0.weight: torch.Size([512, 784])
linear_relu_stack.0.bias: torch.Size([512])
linear_relu_stack.2.weight: torch.Size([512, 512])
linear_relu_stack.2.bias: torch.Size([512])
linear_relu_stack.4.weight: torch.Size([10, 512])
linear_relu_stack.4.bias: torch.Size([10])
Parameter containing:
tensor([[ 0.0330, -0.0246, -0.0207,  ..., -0.0236,  0.0295,  0.0301],
        [-0.0015,  0.0242, -0.0274,  ...,  0.0087, -0.0074, -0.0078],
        [-0.0042, -0.0059, -0.0038,  ..., -0.0006,  0.0291,  0.0335],
        ...,
        [ 0.0198, -0.0322,  0.0142,  ...,  0.0276, -0.0356,  0.0003],
        [-0.0190, -0.0018, -0.0121,  ..., -0.0253,  0.0172,  0.0270],
        [ 0.0267, -0.0019,  0.0036,  ..., -0.0079,  0.0350,  0.0086]],
       requires_grad=True)


In a single training loop, the model makes predictions on the training
dataset (fed to it in batches), and backpropagates the prediction error
to adjust the model\'s parameters.


In [ ]:
# Función de entrenamiento del modelo
# OJO:No se ejecuta nada, solo se define la clase (y sus métodos) para luego crear una instancia del modelo
def train(dataloader, model, loss_fn, optimizer):
    # Obtiene el tamaño total del dataset de entrenamiento
    size = len(dataloader.dataset)
    # Pone el modelo en modo entrenamiento (activa dropout, batch norm, etc.)
    model.train()
    # Itera sobre cada lote (batch) del dataloader
    for batch, (X, y) in enumerate(dataloader):
        # Mueve los datos (imágenes X y etiquetas y) al dispositivo (CPU o GPU)
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        # Realiza la predicción pasando las imágenes por el modelo
        pred = model(X)
        # Calcula el error (loss) comparando las predicciones con las etiquetas reales
        loss = loss_fn(pred, y)

        # Backpropagation
        # Calcula los gradientes de la pérdida respecto a los parámetros del modelo
        loss.backward()
        # Actualiza los parámetros del modelo usando los gradientes calculados
        optimizer.step()
        # Resetea los gradientes a cero para la siguiente iteración
        optimizer.zero_grad()

        # Cada 100 lotes, imprime el progreso del entrenamiento
        if batch % 100 == 0:
            # Obtiene el valor numérico de la pérdida y el número de muestras procesadas
            loss, current = loss.item(), (batch + 1) * len(X)
            # Imprime la pérdida actual y el progreso [muestras_procesadas/total_muestras]
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

We also check the model\'s performance against the test dataset to
ensure it is learning.


In [ ]:
# OJO:No se ejecuta nada, solo se define la clase (y sus métodos) para luego crear una instancia del modelo
def test(dataloader, model, loss_fn):
    # Tamano total del dataset de prueba
    size = len(dataloader.dataset)
    print(f"Test dataset size: {size}")
    # Numero de lotes en el dataloader
    num_batches = len(dataloader)
    print(f"Number of batches: {num_batches}")
    # Modo evaluacion: desactiva dropout/batchnorm en modo entrenamiento
    model.eval()
    # Acumuladores de perdida y aciertos
    test_loss, correct = 0, 0
    # Desactiva el calculo de gradientes para acelerar y ahorrar memoria
    with torch.no_grad():
        for X, y in dataloader:
            # Mueve datos y etiquetas al dispositivo
            X, y = X.to(device), y.to(device)
            # Predicciones del modelo
            pred = model(X)
            # Suma la perdida del lote
            test_loss += loss_fn(pred, y).item()
            # Cuenta aciertos en el lote
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    # Promedia la perdida por lote
    test_loss /= num_batches
    # Calcula la exactitud sobre todo el dataset
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

The training process is conducted over several iterations (*epochs*).
During each epoch, the model learns parameters to make better
predictions. We print the model\'s accuracy and loss at each epoch;
we\'d like to see the accuracy increase and the loss decrease with every
epoch.


In [ ]:
###### Ahora es cuando se ejecuta el entrenamiento y la evaluación del modelo por un número determinado de épocas (epochs) ######
# Numero de epocas de entrenamiento
epochs = 5
# Bucle de entrenamiento y evaluacion por epoca
for t in range(epochs):
    # Muestra el indice de la epoca actual
    print(f"Epoch {t+1}\n-------------------------------")
    # Entrena el modelo con el conjunto de entrenamiento
    train(train_dataloader, model, loss_fn, optimizer)
    # Evalua el modelo con el conjunto de prueba
    test(test_dataloader, model, loss_fn)
# Mensaje final cuando termina el entrenamiento
print("Done!")

Epoch 1
-------------------------------
loss: 2.307548  [   64/60000]
loss: 2.290650  [ 6464/60000]
loss: 2.271974  [12864/60000]
loss: 2.269262  [19264/60000]
loss: 2.233452  [25664/60000]
loss: 2.219121  [32064/60000]
loss: 2.228184  [38464/60000]
loss: 2.194084  [44864/60000]
loss: 2.194654  [51264/60000]
loss: 2.158869  [57664/60000]
Test dataset size: 10000
Number of batches: 157
Test Error: 
 Accuracy: 41.1%, Avg loss: 2.154353 

Epoch 2
-------------------------------
loss: 2.173610  [   64/60000]
loss: 2.157336  [ 6464/60000]
loss: 2.099017  [12864/60000]
loss: 2.114178  [19264/60000]
loss: 2.045093  [25664/60000]
loss: 1.997465  [32064/60000]
loss: 2.030611  [38464/60000]
loss: 1.947553  [44864/60000]
loss: 1.958027  [51264/60000]
loss: 1.881589  [57664/60000]
Test dataset size: 10000
Number of batches: 157
Test Error: 
 Accuracy: 51.9%, Avg loss: 1.881020 

Epoch 3
-------------------------------
loss: 1.924280  [   64/60000]
loss: 1.886587  [ 6464/60000]
loss: 1.769606  [128

Read more about [Training your model](optimization_tutorial.html).


------------------------------------------------------------------------


Saving Models
=============

A common way to save a model is to serialize the internal state
dictionary (containing the model parameters).


In [ ]:
# Guarda los pesos del modelo en un archivo local
torch.save(model.state_dict(), "model.pth")
# Confirma que el archivo se guardo correctamente
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


Loading Models
==============

The process for loading a model includes re-creating the model structure
and loading the state dictionary into it.


In [14]:
# Crea una nueva instancia del modelo en el dispositivo
model = NeuralNetwork().to(device)
# Carga los pesos guardados en el archivo al modelo
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

This model can now be used to make predictions.


In [15]:
# Lista de etiquetas para mapear indices a nombres de clase
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

# Modo evaluacion para desactivar dropout/batchnorm
model.eval()

# Se devuelve una tupla (imagen, etiqueta)
#
# test_data[0][0] devuelve la imagen del primer ejemplo del conjunto de prueba: Un
# torch. Tensor con forma [1, 28, 28] (un canal en escala de grises) y valores float
# normalizados en el rango 0-1 gracias a ToTensor().
# 
# test_data[0][1] devuelve la etiqueta del primer ejemplo del conjunto de prueba:
# Un entero 0-9 (tipo int) que identifica la clase correspondiente a esa imagen.
x, y = test_data[0][0], test_data[0][1]

# No se calculan gradientes durante la inferencia
with torch.no_grad():
    # Envia la imagen al dispositivo
    x = x.to(device)
    # Ejecuta el modelo para obtener logits
    pred = model(x)
    # Obtiene la clase predicha y la clase real
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"


Read more about [Saving & Loading your
model](saveloadrun_tutorial.html).
